# Objective 1 -- Dynamic World Historical Change: Plots

Reads the zonal-statistics CSVs exported by `historical_change_detection.ipynb`
(downloaded from Drive into `outputs/tables/`) and produces the headline QA/results plots.
No Earth Engine calls happen in this notebook -- it is a pure pandas/matplotlib/seaborn
post-processing step on already-exported tables.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config

In [ ]:
sns.set_theme(style="whitegrid")
config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SITE_ORDER = [s["site_id"] for s in config.SITES]
SITE_LABELS = {s["site_id"]: s["site_name"] for s in config.SITES}
PERIOD_ORDER = list(config.DW_PERIODS.keys())  # baseline, pre, current


## Load tables

All 5 CSVs exported by the Objective 1 notebook, read once here. Wide-format class-area
columns (`..._ha_<code>`) are the `class_area_bands()` pattern from that notebook -- one
band per class to avoid a grouped reducer's nested output that doesn't flatten to CSV.


In [ ]:
area_by_period = pd.read_csv(config.TABLES_DIR / "dw_area_by_class_by_site_period.csv")
area_by_year_season = pd.read_csv(config.TABLES_DIR / "dw_area_by_class_by_site_year_season.csv")
prob_by_period = pd.read_csv(config.TABLES_DIR / "dw_mean_probability_by_site_period.csv")
prob_by_year_season = pd.read_csv(config.TABLES_DIR / "dw_mean_probability_by_site_year_season.csv")
transitions = pd.read_csv(config.TABLES_DIR / "dw_transition_area_by_site.csv")


## Plot 1 -- Habitat composition by period, per site

The headline land-cover result: share of each site's area in each `habitat_class`
(see `config.DW_HABITAT_CLASS_LABELS`) across baseline (2016-18) -> pre (2019-21) ->
current (2022-25). Shown as % of classified area rather than raw hectares so sites of very
different sizes (e.g. Mbokishi vs. the corridor phases) are visually comparable.


In [ ]:
habitat_cols = [f"habitat_area_ha_{c}" for c in config.DW_HABITAT_CLASS_CODES]

habitat_long = area_by_period.melt(
    id_vars=["site_id", "period"],
    value_vars=habitat_cols,
    var_name="class_code",
    value_name="area_ha",
)
habitat_long["class_code"] = habitat_long["class_code"].str.extract(r"_(\d+)$").astype(int)
habitat_long["habitat_class"] = habitat_long["class_code"].map(config.DW_HABITAT_CLASS_LABELS)
habitat_long["pct_area"] = habitat_long["area_ha"] / habitat_long.groupby(
    ["site_id", "period"]
)["area_ha"].transform("sum") * 100

class_order = [config.DW_HABITAT_CLASS_LABELS[c] for c in config.DW_HABITAT_CLASS_CODES]
palette = dict(zip(class_order, config.DW_HABITAT_CLASS_VIS["palette"][1:]))

fig, axes = plt.subplots(1, len(SITE_ORDER), figsize=(16, 5), sharey=True)
for ax, site_id in zip(axes, SITE_ORDER):
    site_df = habitat_long[habitat_long["site_id"] == site_id]
    pivot = (
        site_df.pivot(index="period", columns="habitat_class", values="pct_area")
        .reindex(PERIOD_ORDER)[class_order]
    )
    pivot.plot(kind="bar", stacked=True, ax=ax, color=palette, legend=False, width=0.7)
    ax.set_title(SITE_LABELS[site_id])
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
axes[0].set_ylabel("% of classified area")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), title="Habitat class")
fig.suptitle("Habitat class composition by period")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "habitat_composition_by_period.png", dpi=200, bbox_inches="tight")


## Plot 2 -- Conversion pressure composition by period, per site

Same shape as Plot 1 but for `pressure_class` (low/moderate/high, see
`config.DW_PRESSURE_CLASS_LABELS`) -- the direct read on whether conversion pressure is
rising or falling across the corridor-establishment timeline.


In [ ]:
pressure_cols = [f"pressure_area_ha_{c}" for c in config.DW_PRESSURE_CLASS_CODES]

pressure_long = area_by_period.melt(
    id_vars=["site_id", "period"],
    value_vars=pressure_cols,
    var_name="class_code",
    value_name="area_ha",
)
pressure_long["class_code"] = pressure_long["class_code"].str.extract(r"_(\d+)$").astype(int)
pressure_long["pressure_class"] = pressure_long["class_code"].map(config.DW_PRESSURE_CLASS_LABELS)
pressure_long["pct_area"] = pressure_long["area_ha"] / pressure_long.groupby(
    ["site_id", "period"]
)["area_ha"].transform("sum") * 100

pressure_order = [config.DW_PRESSURE_CLASS_LABELS[c] for c in config.DW_PRESSURE_CLASS_CODES]
pressure_palette = dict(zip(pressure_order, config.DW_PRESSURE_VIS["palette"]))

fig, axes = plt.subplots(1, len(SITE_ORDER), figsize=(16, 5), sharey=True)
for ax, site_id in zip(axes, SITE_ORDER):
    site_df = pressure_long[pressure_long["site_id"] == site_id]
    pivot = (
        site_df.pivot(index="period", columns="pressure_class", values="pct_area")
        .reindex(PERIOD_ORDER)[pressure_order]
    )
    pivot.plot(kind="bar", stacked=True, ax=ax, color=pressure_palette, legend=False, width=0.7)
    ax.set_title(SITE_LABELS[site_id])
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
axes[0].set_ylabel("% of classified area")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), title="Conversion pressure")
fig.suptitle("Conversion pressure composition by period")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "pressure_composition_by_period.png", dpi=200, bbox_inches="tight")


## Plot 3 -- Natural-habitat probability trend, 2016-2025

Annual mean `natural_prob` per site (from `dw_mean_probability_by_site_year_season.csv`,
`season == "annual"`) -- a continuous, threshold-free trend line that complements the
discrete `habitat_class` composition above, since it isn't affected by any classification
threshold choice. Mbokishi (the intact reference site) is highlighted as the comparison
baseline for the two conservancies/corridor phases.


In [ ]:
annual = prob_by_year_season[prob_by_year_season["season"] == "annual"].copy()
annual["site_name"] = annual["site_id"].map(SITE_LABELS)

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=annual,
    x="year",
    y="natural_prob_mean",
    hue="site_name",
    hue_order=[SITE_LABELS[s] for s in SITE_ORDER],
    marker="o",
    ax=ax,
)
ax.set_xlabel("Year")
ax.set_ylabel("Mean natural_prob (woody + grass)")
ax.set_title("Natural-habitat probability trend by site (annual composites)")
ax.legend(title="Site")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "natural_prob_trend.png", dpi=200, bbox_inches="tight")


## Plot 4 -- Baseline -> current transition matrix

`transition_area_ha_<from><to>` columns from `dw_transition_area_by_site.csv`
(`comparison == "baseline_to_current"`), normalized to % of each *origin* class's baseline
area -- i.e. "of the area that was class X in baseline, what % is now class Y in current".
The diagonal is persistence; off-diagonal cells are conversion/recovery. Faceted per site so
conversion patterns at the corridor phases can be compared directly against Mbokishi.


In [ ]:
baseline_to_current = transitions[transitions["comparison"] == "baseline_to_current"]
class_labels = config.DW_HABITAT_CLASS_LABELS

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, site_id in zip(axes.flat, SITE_ORDER):
    row = baseline_to_current[baseline_to_current["site_id"] == site_id].iloc[0]
    matrix = pd.DataFrame(
        0.0, index=config.DW_HABITAT_CLASS_CODES, columns=config.DW_HABITAT_CLASS_CODES
    )
    for col in transitions.columns:
        if not col.startswith("transition_area_ha_"):
            continue
        code_pair = col.removeprefix("transition_area_ha_")
        from_code, to_code = int(code_pair[0]), int(code_pair[1])
        matrix.loc[from_code, to_code] = row[col]

    pct_matrix = matrix.div(matrix.sum(axis=1), axis=0) * 100
    pct_matrix.index = pct_matrix.index.map(class_labels)
    pct_matrix.columns = pct_matrix.columns.map(class_labels)

    sns.heatmap(
        pct_matrix,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        vmin=0,
        vmax=100,
        cbar=False,
        ax=ax,
    )
    ax.set_title(SITE_LABELS[site_id])
    ax.set_xlabel("Current (2022-25)")
    ax.set_ylabel("Baseline (2016-18)")

fig.suptitle("Baseline to current habitat-class transitions (% of origin class area)")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "baseline_to_current_transitions.png", dpi=200, bbox_inches="tight")
